In [1]:
import warnings
import pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning) 

from loader import load_data
path="./Final/data"
typeData="csv"


handle_clean=True
handle_tame=True
handle_normalize=True
handle_feature=True
handle_balance=True

y_column='CVD.event'

X,Y,data,all_mappings,y_mappings,weights=load_data(path=f"{path}.{typeData}",y_column=y_column)
X=X.astype("float64")

In [36]:
ratio=pd.Series(Y).value_counts()
ratio[0]/ratio[1]

np.float64(7.8805256869772995)

In [37]:
# Category Based Feature Selection for Expert Opinion
categories=[{"name":"First","num":5,"columns":['age_11',"Sex1","Jobstatu1","joblast1","Degreela1","NIGHTLYS1","PALfinal1","Marriage1","smokingstutus1","BPS_1","BPD_1","AnxietyScore_1","DepressionScore_1"]},
            {"name":"Second","num":3,"columns":['BMI_1', 'Demispan_1','Waistcir_1', 'Hipcir_1', 'Waisthight_1', 'Waisthip_1', 'Midarmcir_1','Height_1', 'Weight_1', 'Height_cm']},
            {"name":"Third","num":3,"columns":['LdL_1', 'Glocuse_1', 'Uricacid_1','Cholesterol_1', 'HSCRP_1', 'HDL_1', 'TRIGLYCERIDES_1']},
            {"name":"Fourth","num":3,"columns":['WBC_1','RBC_1', 'HGB_1', 'HCT_1', 'MCV_1', 'MCH_1', 'MCHC_1', 'PLT_1', 'RDWcv','LYM_Sh_1', 'NEUT_Sh_1', 'PDW_1', 'MPV_1']},
            {"name":"Fourth","num":2,"columns":['J1_1', 'J2_1', 'J3_1','IA1_1', 'IA2_1', 'IA3_1', 'IA4_1', 'IB1_1', 'IB2_1', 'IB3_1', 'IB4_1']}]
categories=[]
cols_to_drop=[]

if(len(categories)==0):
    categories=[{"name": "ALL_COLS","num":0, "columns": X.columns},]
    
X.drop(columns=cols_to_drop,inplace=True)

In [4]:
# import tqdm
# warnings.filterwarnings("ignore", category=tqdm.TqdmWarning)
# from plots import plot_histograms_grouped
# plot_histograms_grouped(X,path)

## Available Algorithms

### Dropping Methods
- Drop columns with null values exceeding threshold  
- Drop rows with missing values  

### Statistical Imputation
- Impute with **mean** or **median**  
- Impute with **class-specific mean/median**  

### Forward & Backward Filling
- **Forward fill (ffill)**  
- **Backward fill (bfill)**  
- **Interpolate** missing values  

### Model-Based Imputation
- **Model Imputation** (predict missing values)  
- **Iterative Model Imputation**  
- **KNN Imputation** (k-nearest neighbors)  

## Validation Data Initialization

In [5]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

In [44]:
import importlib
import clean_data

# reload after editing model_training.py
importlib.reload(clean_data)

<module 'clean_data' from '/home/sinam/Python/AI/qenv/Medical-Framework/clean_data.py'>

In [45]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from clean_data import handling_missing_data
from test_models import train_and_evaluate

if handle_clean:
    x_train_clean, y_train_clean, x_val_clean, y_val_clean, best_algo_clean, best_acc_clean, acc_holder_clean, changes_clean, best_imputer_clean = handling_missing_data(
        x_train, y_train, x_val, y_val, train_and_evaluate, weights
    )
    print(f"Best imputation algorithm: {best_algo_clean} with accuracy: {best_acc_clean}")
    for item in acc_holder_clean:
        print(item, acc_holder_clean[item])
else:
    # Skip cleaning, use original data with same variable names
    x_train_clean = x_train
    y_train_clean = y_train
    x_val_clean = x_val
    y_val_clean = y_val
    best_algo_clean = "No Cleaning"
    best_acc_clean = 0
    acc_holder_clean = {"No Cleaning": 0}
    changes_clean = []
    best_imputer_clean = None
    print("Data cleaning skipped. Using original data.")

Model_imputation: 0.7200
Best imputation algorithm: Model_imputation with accuracy: 0.7199822739258231
Model_imputation 0.7199822739258231


In [8]:
x_val_clean.isna().sum().sum()

np.int64(0)

# Taming Outliers

These functions help clean our dataset from outliers efficiently and selecting the best outlier detection algorithm.

## Available Algorithms  
- **IQR Method(1.5):**  
  - Take 1.5 times the IQR and then subtract this value from Q1 and add this value to Q3


- **LOF:**  
  - For any data object **q**, the **LOF score** is computed as the ratio of the **average local density** of its **k-nearest neighbors** to its **own local density** **[25]**.  

  $$
  LOF(q) = \frac{\sum_{x \in N_k(q)} lrd(x)}{|N_k(q)| \times lrd(q)}
  $$

  where the **local reachability density (lrd)** of **q** is given by:  

  $$
  lrd(q) = \frac{|N_k(q)|}{\sum_{x \in N_k(q)} \max(\text{dist}_k(x, D), \text{dist}(q, x))}
  $$

- **SP:**  
   - Employ a **scoring measure** based on the nearest neighbor (**k = 1**) within random sub-samples (**S ⊂ D**).  

    $$  S_p(q) = \min_{{x \in S}} \text{dist}(q, x) $$

    where **dist(q, x)** represents the distance between **q** and **x**.

- **iForest:**  
  - A **random split** is performed on a randomly selected feature.  
  - The partitioning continues until either:  
    - Each node contains only **one data object**, or  
    - The tree reaches its **height limit**.

  $$
  iForest(q) = \frac{1}{t} \sum_{i=1}^{t} l_i(q)
  $$


- **iNNe:**  
  - This method builds **hyperspheres** using all dimensions of the dataset. The **isolation score** of a data object **q** is defined as:  

  $$
  I(q) =
  \begin{cases}
  \tau (\eta_{cnn}(q)), & \text{if } q \in \bigcup_{c \in S} B(c) \\  
  1 - \tau (cnn(q)), & \text{otherwise}  
  \end{cases}
  $$


In [9]:
import importlib
import tame_outlier
# reload after editing model_training.py
importlib.reload(tame_outlier)

<module 'tame_outlier' from '/home/sinam/Python/AI/qenv/Medical-Framework/tame_outlier.py'>

In [10]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from tame_outlier import taming_outliers

if handle_tame:
    x_train_tame, y_train_tame, x_val_tame, y_val_tame, best_algo_tame, best_acc_tame, acc_holder_tame, changes_tame, best_imputer_tame = taming_outliers(
        x_train_clean, y_train_clean, x_val_clean, y_val_clean,weights
    )
    print(f"Best Taming Outlier algorithm: {best_algo_tame} with accuracy: {best_acc_tame}")
    for item in acc_holder_tame:
        print(item, acc_holder_tame[item])
else:
    # Skip outlier taming, use cleaned data with same variable names
    x_train_tame = x_train_clean
    y_train_tame = y_train_clean
    x_val_tame = x_val_clean
    y_val_tame = y_val_clean
    best_algo_tame = "No Taming"
    best_acc_tame = 0
    acc_holder_tame = {"No Taming": 0}
    changes_tame = []
    best_imputer_tame = None
    print("Outlier taming skipped. Using cleaned data.")

Fit Complete: 2 columns for Log, 8 for Flagging.
Outlier Phase: Created 8 flags and logged 2 columns.
Best Taming Outlier algorithm: Medical_Log_Flag with accuracy: 0.0
Medical_Log_Flag 1.0


# Normalization Algorithms

These functions help standardize or normalize datasets to improve model performance and feature scaling.

## Available Algorithms

### 1. MinMaxScalarNorm
- **Description:**
  - Scales features to a fixed range [0, 1] using minimum and maximum values
  - Formula:
    $$
    X_norm = (X - X_min ) / (X_max - X_min)
    $$


### 2. RobustScalarNorm
- **Description:**
  - Scales features using median and interquartile range (IQR) to handle outliers
  - Formula:
    $$
    X_robust = (X - Median(X)) / IQR(X)
    $$


### 3. ZScoreNormalizationNorm
- **Description**:
  - Standardizes features to have zero mean and unit variance
  - Formula:
    $$
    X_std = (X - μ) / σ
    $$


In [11]:
import importlib
import normalization
# reload after editing model_training.py
importlib.reload(normalization)

<module 'normalization' from '/home/sinam/Python/AI/qenv/Medical-Framework/normalization.py'>

In [12]:
if(handle_normalize):
    from normalization import normalization
    x_train_normalize, y_train_normalize,x_val_normalize, y_val_normalize, best_algo_normalization, best_acc_normalization,acc_holder_normalization,best_imputer_normalization=normalization(x_train_tame,y_train_tame,x_val_tame,y_val_tame,weights)
    for item in acc_holder_normalization:
        print(item,acc_holder_normalization[item])
else:
    x_train_normalize=x_train_tame.copy()
    y_train_normalize=y_train_tame.copy()
    x_val_normalize=x_val_tame.copy()
    y_val_normalize=y_val_tame.copy()
    best_algo_normalization="Not Normalized"

Selecting best normalization algorithm...
MinMaxScalerNorm 0.7159258845744407
RobustScalerNorm 0.7159258845744407
ZScoreNormalizationNorm 0.7159258845744407
PowerTransformerNorm 0.7228096631213561


In [13]:
x_val_normalize.isna().sum().sum()

np.int64(0)

### Some Plots
##### 1.Plotting Y Column With Pie Chart to View Balance Between Classes
##### 2.Heatmap for Correlation
##### 3.Histogram for After Normalized

In [14]:
# from plots import plot_pie_chart
# plot_pie_chart(pd.Series(Y),path=path,title="Y Column Pie Chart")

In [15]:
# # Plot Correlation Heatmap
# from plots import plot_heatmap
# plot_heatmap(x_train_normalize,pd.Series(y_train_normalize,name="Y"),f"{path}_before_")
# plot_histograms_grouped(x_train_normalize,path=f"{path}_after_")

# Feature Selection Algorithms

These functions help select the most relevant features from a dataset to improve model performance and reduce dimensionality.

## Available Algorithms

---

### 1. $ SelectK $
- **Description:**
  - Selects the top K features based on statistical tests (e.g., `chi2`, `f_classif`)
  - Uses univariate statistical tests to score each feature

- **Parameters:**
  - `k`: Number of top features to select  
  - `score_func`: Scoring function (default: `f_classif`)

---

### 2. $ L_1 Based $
- **Description:**
  - Selects features using L1-regularized linear models (e.g., Lasso, Logistic Regression)
  - Features with non-zero coefficients are selected

- **Parameters:**
  - `estimator`: L1-regularized model (e.g., `LogisticRegression`, `Lasso`)  
  - `threshold`: Minimum coefficient value for selection

---

### 3. $ TreeBased $
- **Description:**
  - Selects features based on importance scores from tree-based models
  - Uses the `feature_importances_` attribute from tree estimators

- **Parameters:**
  - `estimator`: Tree-based model (e.g., `RandomForest`, `XGBoost`)  
  - `threshold`: Minimum importance score for selection

---

### 4. $ RFE (Recursive Feature Elimination) $
- **Description:**
  - Recursively removes least important features using a base model
  - Fits the model multiple times and eliminates features with the smallest weights

- **Parameters:**
  - `estimator`: Model used to evaluate feature importance (default: `LogisticRegression`)  
  - `n_features_to_select`: Number of features to retain

---

### 5. $ Chi2 $
- **Description:**
  - Selects top K features using the Chi-squared statistical test
  - Suitable for non-negative, categorical or count data

- **Parameters:**
  - `k`: Number of top features to select

---

### 6. $ VarianceThreshold $
- **Description:**
  - Removes features with low variance, assuming low-variance features do not carry useful information
  - Filters features below a specified threshold

- **Parameters:**
  - `threshold`: Minimum variance required to keep a feature



In [54]:
import importlib
import feature_selection
# reload after editing model_training.py
importlib.reload(feature_selection)

<module 'feature_selection' from '/home/sinam/Python/AI/qenv/Medical-Framework/feature_selection.py'>

In [55]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from feature_selection import feature_selection

if handle_feature:
    x_train_feature, y_train_feature, x_val_feature, y_val_feature, best_algo_feature, best_acc_feature, acc_holder_feature, final_scores_feature, final_results_feature,kmeans_feature = feature_selection(
        x_train_normalize, y_train_normalize, x_val_normalize, y_val_normalize, categories,weights
    )
    print(f"Best Feature Selection algorithm: {best_algo_feature} with accuracy: {best_acc_feature}")
    for item in acc_holder_feature:
        print(item, acc_holder_feature[item])
else:
    # Skip feature selection, use normalized data with same variable names
    x_train_feature = x_train_normalize
    y_train_feature = y_train_normalize
    x_val_feature = x_val_normalize
    y_val_feature = y_val_normalize
    best_algo_feature = "No Feature Selection"
    best_acc_feature = 0
    acc_holder_feature = {"No Feature Selection": 0}
    final_scores_feature = None
    final_results_feature = "Feature selection skipped. Using all features."
    kmeans_feature=None
    print("Feature selection skipped. Using all features.")

Category 'ALL_COLS': Picking top 42 features (RFECV Optimized)
Final Selection Complete: 57 features total.
Best Feature Selection algorithm: Hybrid_Category_RFECV with accuracy: 0


In [18]:
len(x_train_feature.columns)

57

##### Correlation heatmap After Feature Selection

In [19]:
# plot_heatmap(x_train_feature,pd.Series(y_train_feature,name="Y"),f"{path}_after_")

##### Balance Sample

In [20]:
import importlib
import balance
# reload after editing model_training.py
importlib.reload(balance)

<module 'balance' from '/home/sinam/Python/AI/qenv/Medical-Framework/balance.py'>

In [21]:
from balance import check_and_balance
if handle_balance:
    x_train_bal, y_train_bal, used_method = check_and_balance(
        x_train_feature, y_train_feature,x_val_feature, y_val_feature,weights
    )
    print(f"Balancing method used: {used_method}")
else:
    # Skip balancing, use feature-selected data with same variable names
    x_train_bal = x_train_feature
    y_train_bal = y_train_feature
    used_method = "No Balancing"
    print("Data balancing skipped. Using original data.")

Data is already balanced. Skipping balancing step.
Balancing method used: none


##### Export Clean data with No Outliers, Normalizad and Balanced

In [22]:
x_full = pd.concat([x_train_bal, x_val_feature], axis=0).reset_index(drop=True)
y_full = pd.concat([pd.Series(y_train_feature,name='Y train'), pd.Series(y_val_feature,name='Y train')], axis=0).reset_index(drop=True)
full_data = x_full.copy()
full_data[y_column] = y_full
full_data.to_csv(f'{path}_balanced{"_and_normalized" if handle_normalize else ""}.csv')

# Model Training Algorithms

These functions train and evaluate machine learning models with hyperparameter tuning using **GridSearchCV**. Each algorithm runs multiple configurations, evaluates performance using a custom scoring function, and selects the best-performing model.  

## Available Algorithms

---

### 1. $ LogisticRegressionBased $
- **Description:**
  - Uses Logistic Regression with elastic net regularization for classification.
  - Handles imbalanced datasets using `class_weight='balanced'`.
  - Useful for both feature selection and baseline classification.

- **Parameters (Grid Search):**
  - `penalty`: [`l1`, `l2`, `elasticnet`]  
  - `C`: [`0.001`, `0.01`, `0.1`, `1`, `10`]  
  - `solver`: [`saga`]  
  - `max_iter`: [`1000`]  
  - `l1_ratio`: [`0`, `0.5`, `1`]  

---

### 2. $ KNNBased $
- **Description:**
  - Uses the K-Nearest Neighbors algorithm.
  - Classifies data based on the majority class of nearest neighbors.

- **Parameters (Grid Search):**
  - `n_neighbors`: [`3`, `5`, `7`, `9`]  
  - `weights`: [`uniform`, `distance`]  
  - `algorithm`: [`auto`, `ball_tree`, `kd_tree`]  

---

### 3. $ NaiveBayesBased $
- **Description:**
  - Uses Gaussian Naive Bayes for classification.
  - Suitable for continuous features and fast baseline performance.

- **Parameters (Grid Search):**
  - `var_smoothing`: [`1e-9`, `1e-8`, `1e-7`]  

---

### 4. $ RandomForestBased $
- **Description:**
  - Uses an ensemble of decision trees with bagging.
  - Selects features based on importance scores and supports imbalanced datasets.

- **Parameters (Grid Search):**
  - `n_estimators`: [`50`, `100`]  
  - `max_depth`: [`None`, `10`, `20`]  
  - `min_samples_split`: [`2`, `5`]  

---

### 5. $ XGBoostBased $
- **Description:**
  - Gradient boosting framework optimized for speed and performance.
  - Supports imbalanced learning with `scale_pos_weight`.

- **Parameters (Grid Search):**
  - `n_estimators`: [`50`, `100`]  
  - `max_depth`: [`3`, `6`, `10`]  
  - `learning_rate`: [`0.01`, `0.1`, `0.2`]  

---

### 6. $ LightGBMBased $
- **Description:**
  - Gradient boosting framework optimized for speed and efficiency.
  - Handles categorical features and imbalanced data well.

- **Parameters (Grid Search):**
  - `n_estimators`: [`100`, `200`]  
  - `learning_rate`: [`0.01`, `0.1`]  
  - `num_leaves`: [`31`, `63`]  
  - `max_depth`: [`-1`, `3`, `5`, `10`, `20`]  

---

### 7. $ CatBoostBased $
- **Description:**
  - Gradient boosting library that handles categorical features automatically.
  - Does not require one-hot encoding.

- **Parameters (Grid Search):**
  - `iterations`: [`100`, `200`]  
  - `learning_rate`: [`0.01`, `0.1`]  
  - `depth`: [`4`, `6`, `8`]  

---

### 8. $ GradientBoostingBased $
- **Description:**
  - Classic gradient boosting implementation in scikit-learn.
  - Suitable for smaller datasets and interpretable models.

- **Parameters (Grid Search):**
  - `n_estimators`: [`50`, `100`]  
  - `learning_rate`: [`0.01`, `0.1`]  
  - `max_depth`: [`3`, `5`]  

---

### 9. $ NeuralNetworkBased $
- **Description:**
  - Multi-layer Perceptron (MLP) classifier.
  - Learns complex non-linear decision boundaries.

- **Parameters (Grid Search):**
  - `hidden_layer_sizes`: [`(50,)`, `(100,)`, `(50, 50)`, `(100, 50)`]  
  - `activation`: [`relu`, `tanh`]  
  - `solver`: [`adam`]  
  - `max_iter`: [`300`]  

---

### 10. $ SVMBased $
- **Description:**
  - Support Vector Machine (SVM) classifier with kernel support.
  - Suitable for high-dimensional data and non-linear boundaries.

- **Parameters (Grid Search):**
  - `C`: [`0.1`, `1`, `10`]  
  - `kernel`: [`linear`, `rbf`]  
  - `gamma`: [`scale`, `auto`]  


In [23]:
import importlib
import model_training

# reload after editing model_training.py
importlib.reload(model_training)

<module 'model_training' from '/home/sinam/Python/AI/qenv/Medical-Framework/model_training.py'>

In [24]:
x_train_feature

,Jobstatu1,Waisthight_1,FileNo,Patient_Phenotype_1,HGB_1,Glocuse_1_X_age_11,MCHC_1,PALfinal1,RDWcv,IA4_1,...,NEUT_Sh_1,Waistcir_1,WBC_1,BPS_1_X_J2_1,J2_1,BPD_1,Demispan_1,TRIGLYCERIDES_1,J1_1_X_J2_1,Cholesterol_1
0,0.529350,-0.268089,1.115451,False,-0.804216,-0.621832,-1.680850,0.445852,1.457244,0.241872,...,-0.320138,-0.318104,-0.757746,-0.301795,0.241361,-0.846792,-0.303130,-0.994878,0.105710,-1.295644
1,0.529350,1.141722,0.505699,False,-0.557938,-1.467232,-0.657075,-0.554599,1.030893,0.241872,...,-1.067342,1.246478,-1.580720,0.017547,0.241361,0.156909,-0.858462,-0.826160,0.105710,0.863937
2,0.529350,-1.249856,1.278457,True,-0.241374,0.328735,-0.015019,-1.403123,-0.192567,0.241872,...,0.188444,-0.833439,1.134022,-0.089427,-2.706804,0.005933,0.150521,-0.182921,-1.185512,-1.361850
3,-1.160489,-0.684020,-0.522315,True,0.626703,0.087854,1.002248,-1.072684,-0.663792,0.241872,...,0.369662,-0.531459,0.286165,-0.103504,0.241361,0.335465,-0.303130,-0.623577,0.105710,0.052766
4,1.865308,-0.824196,-0.921330,True,0.285164,0.032405,-0.850462,-1.351314,0.534629,0.241872,...,-0.004736,-0.233291,0.081729,0.007974,0.241361,0.156909,0.947530,1.322417,0.105710,-0.215707
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5941,0.529350,-1.344475,0.052722,False,0.836203,0.606303,0.641100,0.228668,-0.365945,0.241872,...,-0.320138,-1.539134,-0.842275,-0.465525,0.241361,-1.881159,-0.303130,-0.696839,0.105710,-0.243218
5942,0.529350,-0.251533,0.063369,False,0.488942,1.249588,-0.383550,0.469393,0.818308,0.241872,...,-0.004736,-0.445888,0.849227,-0.147091,0.241361,-0.781297,-1.140830,0.798026,0.105710,1.478640
5943,0.529350,1.288035,0.188532,False,-0.432472,0.009692,-0.166095,0.617418,-0.512048,0.241872,...,-1.067342,1.246478,-0.280863,0.017547,0.241361,1.406607,-0.029974,0.294919,0.105710,0.459290
5944,1.865308,0.059453,-0.723772,True,1.265459,0.635181,1.096183,-1.311214,-1.098220,-2.979879,...,-0.106250,0.186424,0.481768,0.151286,0.241361,1.778121,0.684556,1.773047,0.105710,0.955475


In [25]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from model_training import model_training

model, best_algo_model, best_accuracy_model,acc_holder_model,cm,fpr,tpr,total_combinations_across_all_models=model_training(x_train_feature,x_val_feature,pd.Series(y_train_feature,name="Y train"),pd.Series(y_val_feature,name="Y val"),weights)

print(f"Best Model: {best_algo_model} with accuracy: {best_accuracy_model} amongst {total_combinations_across_all_models} configs.")

for item in acc_holder_model:
        print(item,acc_holder_model[item])

Running LogisticRegression with 45 model configurations...
LogisticRegression ROC-AUC Score: 0.7690
Running KNeighbors with 24 model configurations...
KNeighbors ROC-AUC Score: 0.6282
Running GaussianNB with 3 model configurations...
GaussianNB ROC-AUC Score: 0.7335
Running RandomForestBased with 12 model configurations...
RandomForestBased ROC-AUC Score: 0.7424
Running XGBoostBased with 18 model configurations...
XGBoostBased ROC-AUC Score: 0.7251
Running LogisticRegression with 45 model configurations...
LogisticRegression ROC-AUC Score: 0.7690
Running RandomForestBased with 12 model configurations...
RandomForestBased ROC-AUC Score: 0.7424
Running GaussianNB with 3 model configurations...
GaussianNB ROC-AUC Score: 0.7335
Running XGBoostBased with 18 model configurations...
XGBoostBased ROC-AUC Score: 0.7251
Running KNeighbors with 24 model configurations...
KNeighbors ROC-AUC Score: 0.6282
FINAL CALIBRATED STACKED ROC-AUC: 0.7836
Best Model: StackedGeneralization with accuracy: 0.78

In [26]:
from export import save_pipeline_summary
norm="normalized" if handle_normalize else ""
save_pipeline_summary(final_results_feature,model,best_algo_clean,best_algo_tame,best_algo_normalization,best_algo_feature,best_algo_model,best_accuracy_model,file_path=f"./{path}_logs_{norm}.txt")

Summary saved to ././Final/data_logs_normalized.txt


In [27]:
import joblib
import cloudpickle
import os
path_to_dir = path.split("/")
path_to_dir.pop()
new_path = "/".join(path_to_dir)

joblib.dump(model, os.path.join(new_path, "model.pkl"))

imputer = best_imputer_clean

with open(os.path.join(new_path, "clean_data.pkl"), "wb") as f:
    cloudpickle.dump(imputer, f)

joblib.dump(all_mappings, os.path.join(new_path, "all_mapping.pkl"))

joblib.dump(y_mappings, os.path.join(new_path, "y_mappings.pkl"))

joblib.dump(x_train_feature.columns.tolist(), os.path.join(new_path, "features.pkl"))

['./Final/features.pkl']

In [28]:
from plots import calculate_p_values,format_p_values_table,SHAP,plot_confusion_matrix_roc
p_values_df=calculate_p_values(x_train_normalize,y_train_normalize)
SHAP(x_train_normalize,y_train_normalize,path)
plot_confusion_matrix_roc(fpr,tpr,cm,path)
format_p_values_table(p_values_df,path)

/home/sinam/Python/miniconda3/envs/qenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
 99%|===================| 1173/1190 [00:30<00:00]        

,Feature,P-Value,Significance
25,Glocuse_1,3.63e-40,Significant
1,age_11,5.92e-40,Significant
44,J1_1,9.17e-37,Significant
10,BPS_1,2.47e-30,Significant
45,J2_1,4.37e-29,Significant
57,Glocuse_1_extreme_flag,1.46e-22,Significant
46,J3_1,1.52e-21,Significant
11,BPD_1,1.38e-15,Significant
30,TRIGLYCERIDES_1,1.84e-15,Significant
19,Waisthip_1,2.16e-13,Significant
